# Tutorial: Removing for-loops and making your numpy code more readable

## 1. Matrix Multiplication (and similar things)

Lets say we want to multiply matrices that exist in some grid with arbitrary size.

The most intuitive way to do this is to make a $\texttt{for}$ loop that loops through the grid:

In [ ]:
import numpy as np

H, W, i, j, k = 125, 133, 3, 4, 5 # Some random numbers

def slow_for_loop(A, B):
    C = np.zeros((H, W, i, k))
    for h in range(H):
        for w in range(W):
            C[h, w] = A[h, w] @ B[h, w]
    return C

In [ ]:
A = np.random.randn(H, W, i, j)
B = np.random.randn(H, W, j, k)

Below, we will look how long this function takes to run

In [ ]:
%timeit slow_forloop(A, B)

This is not a good time, especially, if it has to be done many times in your program.
Even worse, if either of the matrices happens to be ordered incorrectly (e.g. if B was of shape (H, W, k, j)), the above function would immediately fail.

But no worry! We can use $\texttt{einsum}$ instead!

In [ ]:
%timeit np.einsum('hwij, hwjk -> hwik', A, B)

Here, the string we supply to the $\texttt{einsum}$ function describes exactly the operation that we apply to the inputs.

In this case, we say that A has the dimensions (h, w, i, j) and B has (h, w, j, k) and we simply say that the output has dimension (h, w, i, k). \
This is interpreted as a multiplication of A and B where we sum in all the dimensions that are not present in the output. \
Since "j" is not present in the output, we know that this dimension has been summed out. \
Note that this is exactly the same as matrix multiplication in each (h, w)!

## 2. More advanced linear algebra operations

Luckily, the more advanced operations are automatically batched in numpy.
Lets look at some!

### Determinant

In [ ]:
m, n = 3, 3
C = np.random.randn(H, W, m, n)
det = np.linalg.det(C)
print(det.shape)

### Inverse/Solving systems of equations

In [ ]:
d = np.random.randn(H, W, n)
e = np.einsum('hwmn, hwn -> hwm', C, d)
Cinv = np.linalg.inv(C)
d_1 = np.einsum('hwnm, hwm -> hwn', Cinv, e)
d_2 = np.linalg.solve(C, e)

print(np.linalg.norm(d - d_1)) # Approx zero
print(np.linalg.norm(d - d_2)) # Approx zero

### Eigenvalues/Eigenvectors

In [ ]:
lam, V = np.linalg.eig(C)
print(lam.shape)
print(V.shape)

### Trace

For some reason, the authors of numpy decided that this function should use a completely different syntax.

In [ ]:
tr1 = np.trace(C, axis1=-2, axis2=-1)
print(tr1.shape)

# We can also use the einsum function
tr2 = np.einsum('hwpp -> hw', C)
print(tr2.shape)
print(np.linalg.norm(tr1 - tr2))

## 3. Case study: A difficult operation

Let's say we have a symmetric matrix on each point of a grid. \
We want to change these matrices. \
Instead, we want to have matrices with the same eigenvectors, but whose eigenvalues should be the exponentiated negated eigenvalues of the original matrix. \
Doing this can be quite tricky without $\texttt{for}$ loops, let's give it a try!

In [ ]:
N = 10
J = np.random.randn(N, H, W, 2)
T = np.einsum('nhwd, nhwD -> hwdD', J, J) # Create a symmetric tensor by summing the outer products of some vectors
lam, V = np.linalg.eig(T)
lam = np.exp(-lam)
# Recall that T = V @ lam @ V^T since T is symmetric
T_2 = np.einsum('hwij, hwj, hwkj -> hwik', V, lam, V)

And that's all we needed to do! To recap the code above. 
1. We created a symmetric matrix in every point by summing together some outer products of vectors.
2. We took the eigendecomposition in every point
3. We exponeniated the negated values of our matrix
4. We put the matrix back together by matrix multiplication using $\texttt{einsum}$

## 4. Final Notes

We have looked at some standardized, vectorized Linear Algebra operations in $\texttt{numpy}$, and shown that you can do almost any computation without having to resort to nasty $\texttt{for}$ loops :)

However, learning this syntax takes time. \
So, the recommendation is to just trying out a bunch of these types of operations, as well as checking out some other blog posts on this subject:

More info on einsum: \
https://ajcr.net/Basic-guide-to-einsum/ \
and: https://rockt.github.io/2018/04/30/einsum